# Resultados Finais — Pipeline de Identificação de Biomarcadores

Consolida todas as métricas, visualizações e os biomarcadores identificados
para apresentação no TCC.

In [ ]:
import os, json, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.metrics import (
    roc_curve, precision_recall_curve, roc_auc_score,
    average_precision_score, confusion_matrix, ConfusionMatrixDisplay
)

sns.set_theme(style='whitegrid')
PROC    = '../data/processed'
MODELS  = '../models'
REPORTS = '../reports'
print('Pronto.')

## 1. Tabela comparativa de modelos

In [ ]:
with open(os.path.join(REPORTS, 'metrics.json')) as f:
    metrics = json.load(f)

df_metrics = pd.DataFrame(metrics).T.reset_index().rename(columns={'index': 'Modelo'})
df_metrics.columns = ['Modelo', 'Accuracy', 'Precision', 'Recall', 'F1', 'AUC-ROC', 'AUC-PRC']
df_metrics = df_metrics.sort_values('AUC-ROC', ascending=False).reset_index(drop=True)

print('=== Comparação de Modelos ===')
display(df_metrics.style
    .highlight_max(subset=['Accuracy','Precision','Recall','F1','AUC-ROC','AUC-PRC'],
                   color='#c8e6c9')
    .format({
        'Accuracy': '{:.4f}', 'Precision': '{:.4f}', 'Recall': '{:.4f}',
        'F1': '{:.4f}', 'AUC-ROC': '{:.4f}', 'AUC-PRC': '{:.4f}'
    }))

## 2. Curvas ROC e PRC — Melhor modelo

In [ ]:
X_test = np.load(os.path.join(PROC, 'X_test.npy'))
y_test = np.load(os.path.join(PROC, 'y_test.npy'))

with open(os.path.join(MODELS, 'best_model_meta.json')) as f:
    meta = json.load(f)
best_name = meta['best_model_name']

with open(os.path.join(MODELS, 'best_model.pkl'), 'rb') as f:
    best_model = pickle.load(f)

y_prob = best_model.predict_proba(X_test)[:, 1]
print(f'Melhor modelo: {best_name}')
print(f'AUC-ROC: {roc_auc_score(y_test, y_prob):.4f}')
print(f'AUC-PRC: {average_precision_score(y_test, y_prob):.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ROC
fpr, tpr, _ = roc_curve(y_test, y_prob)
auc_roc = roc_auc_score(y_test, y_prob)
axes[0].plot(fpr, tpr, linewidth=2.5, color='#2c7bb6', label=f'AUC = {auc_roc:.3f}')
axes[0].plot([0,1],[0,1],'k--', linewidth=1)
axes[0].set_xlabel('Taxa de Falsos Positivos', fontsize=12)
axes[0].set_ylabel('Taxa de Verdadeiros Positivos', fontsize=12)
axes[0].set_title(f'Curva ROC — {best_name}', fontsize=13)
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# PRC
prec, rec, _ = precision_recall_curve(y_test, y_prob)
auc_prc = average_precision_score(y_test, y_prob)
axes[1].plot(rec, prec, linewidth=2.5, color='#d7191c', label=f'AP = {auc_prc:.3f}')
axes[1].set_xlabel('Recall', fontsize=12)
axes[1].set_ylabel('Precision', fontsize=12)
axes[1].set_title(f'Curva PRC — {best_name}', fontsize=13)
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.suptitle('Avaliação do Melhor Modelo', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(REPORTS, 'best_model_roc_prc.png'), dpi=150)
plt.show()

## 3. Matriz de Confusão

In [ ]:
y_pred = best_model.predict(X_test)
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                               display_labels=['Negativo', 'Positivo (marcador)'])
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title(f'Matriz de Confusão — {best_name}', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(REPORTS, 'confusion_matrix.png'), dpi=150)
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f'TP={tp}  FP={fp}  TN={tn}  FN={fn}')
print(f'Sensibilidade (Recall): {tp/(tp+fn):.4f}')
print(f'Especificidade: {tn/(tn+fp):.4f}')

## 4. SHAP Beeswarm Plot

In [ ]:
shap_img = os.path.join(REPORTS, 'shap_summary.png')
if os.path.exists(shap_img):
    from IPython.display import Image, display
    display(Image(shap_img))
else:
    print('Execute o módulo 05_shap.py primeiro para gerar este plot.')

## 5. Top 20 K-mers — Biomarcadores Identificados

In [ ]:
csv_path = os.path.join(REPORTS, 'top_kmers.csv')
if os.path.exists(csv_path):
    df_kmers = pd.read_csv(csv_path)
    print(f'Top {len(df_kmers)} k-mers identificados como potenciais biomarcadores:\n')
    display(df_kmers.style
        .background_gradient(subset=['shap_importance'], cmap='Greens')
        .format({'shap_importance': '{:.6f}'}))
else:
    print('Execute o módulo 05_shap.py primeiro.')

In [ ]:
# Bar chart dos top k-mers
bar_img = os.path.join(REPORTS, 'top_kmers_bar.png')
if os.path.exists(bar_img):
    from IPython.display import Image, display
    display(Image(bar_img))
else:
    print('Execute o módulo 05_shap.py primeiro.')

## 6. Discussão: O que esses motivos de aminoácidos indicam?

Os k-mers com maior importância SHAP representam **trímeros de aminoácidos**
que são sistematicamente mais frequentes (ou menos frequentes) em proteínas
marcadoras em comparação às não-marcadoras.

### Interpretação biológica

- **Motivos enriquecidos em positivos**: se os top k-mers contêm aminoácidos
  carregados (D, E, K, R) ou aromáticos (F, Y, W), isso sugere regiões de
  interação proteína-proteína, sítios de fosforilação ou domínios funcionais
  conservados.

- **Motivos hidrofóbicos** (A, V, I, L, M): podem indicar hélices
  transmembrana ou núcleos hidrofóbicos estáveis — estruturas típicas de
  marcadores de membrana.

- **Prolina (P)** em sequências curtas: frequentemente associada a regiões
  de dobramento rígido e sítios de ligação a anticorpos.

### Limitações e próximos passos

1. A representação k-mer ignora a **ordem posicional** dos motivos — futuras
   versões podem usar encodings posicionais ou redes neurais convolucionais.
2. Validação experimental dos top k-mers é necessária para confirmar o papel
   biológico como biomarcadores.
3. Comparação com bases de dados de motivos conservados (PROSITE, InterPro)
   pode revelar sobreposição com domínios funcionais conhecidos.